### imports and pandas settings

In [8]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [9]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [10]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [11]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "baseline" / "RandomForest_thres0.9_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [12]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [13]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [14]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [15]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.46,,,,,
1,0,cf_1,,,,7,,,18.9745,,,0.1254,2,False,0.0633,0.07
2,0,cf_2,3,,,,,,18.9619,,,0.1258,2,False,0.0633,0.08
3,0,cf_3,,,,7,,,18.895,,,0.1281,2,False,0.0633,0.0733
4,0,cf_4,,,6,,,,18.3612,,,0.146,2,False,0.0633,0.04
5,0,cf_5,,,,,,,18.3468,5,,0.1108,2,False,0.0633,0.05
6,0,cf_6,,,,,,,15.265,,,0.125,1,False,0.0633,0.17
7,0,cf_7,3,,,,,,17.3912,,,0.1786,2,False,0.0633,0.0667
8,0,cf_8,,,,6,,,17.2471,,,0.1418,2,False,0.0633,0.2067
9,0,cf_9,,,6,7,,,18.9745,,,0.2504,3,False,0.0633,0.0933


### Filtering Valid CFs

In [16]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.46,,,,,
9,0,cf_10,,,,7,,,18.9745,7,,0.2504,3,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,2.52,,,,,
10,1,cf_3,,,2,,1,,22.3757,,,0.1713,3,True,0.1467,0.0233
11,1,cf_5,,,,,1,,22.3757,1,,0.1475,3,True,0.1467,0.05
12,1,cf_6,,,,,1,,22.3757,5,,0.2189,3,True,0.1467,0.05
13,1,cf_7,2,,,,1,,22.3757,,,0.2546,3,True,0.1467,0.01
14,1,cf_9,,,,3,1,,22.3757,,,0.1546,3,True,0.1467,0.0
2,2,xin,5,3,1,1,4,0,22.694,7,2.64,,,,,
15,2,cf_2,3,,,,2,,22.68,,,0.1673,3,True,0.1567,0.0133


### select one CF

In [17]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.46,,,,,
9,0,cf_10,,,,7,,,18.9745,7,,0.2504,3,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,2.52,,,,,
10,1,cf_5,,,,,1,,22.3757,1,,0.1475,3,True,0.1467,0.05
2,2,xin,5,3,1,1,4,0,22.694,7,2.64,,,,,
11,2,cf_2,3,,,,2,,22.68,,,0.1673,3,True,0.1567,0.0133
3,3,xin,3,3,6,6,2,0,24.3418,1,2.87,,,,,
12,3,cf_1,,,,,,,24.3375,,,0.0008,1,True,0.02,0.01
4,4,xin,5,4,2,7,1,0,21.2585,3,2.94,,,,,
13,4,cf_5,,3,,,,,21.2585,,,0.2368,2,True,0.0267,0.0133


In [18]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_10 ---
Predicted risk: 0.03
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_3 ---
Predicted risk: 0.0233
Valid: True
Changes:
  cgtsmok: 1 → 2
  slprl: 3 → 1

--- cf_5 ---
Predicted risk: 0.05
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_6 ---
Predicted risk: 0.05
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 5

--- cf_7 ---
Predicted risk: 0.01
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1

--- cf_9 ---
Predicted risk: 0.0
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1


=